In [107]:
from astropy.io import fits
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import emcee
from find_source import summary
import corner
import math
from astropy.coordinates import Angle
import astropy.units as units
import scipy.special as sp
import warnings
import itertools

In [108]:
def p_model(p_params, u, v):
    i0, l0, m0 = p_params
    return i0 * np.exp(-2*np.pi*1j*(u*l0 + v*m0))

def c_model(c_params, u, v):
    s0, l0, m0, vis_sigma = c_params
    return s0 * np.exp(-0.5*(u**2 + v**2)/vis_sigma**2) * np.exp(-2*np.pi*1j*(u*l0 + v*m0))

def g_model(g_params, u, v):
    s0, l0, m0, vis_sigma, ratio, vis_theta = g_params
    return s0 * np.exp(-0.5*((u*np.cos(vis_theta)-v*np.sin(vis_theta))**2 + (u*np.sin(vis_theta)+v*np.cos(vis_theta))**2/ratio**2)/vis_sigma**2) \
            * np.exp(-2*np.pi*1j*(u*l0 + v*m0))

def d_model(d_params, u, v):
    s0, l0, m0, vis_r = d_params
    return s0 * 2*vis_r/np.sqrt(u**2+v**2) * sp.j1(np.sqrt(u**2+v**2)/vis_r) * np.exp(-2*np.pi*1j*(u*l0 + v*m0))

In [109]:
def p_p0(peak, rad_coord, rad_pix, rad_barea, total_flux, n_walkers):
    p0 = np.zeros((n_walkers, 3))
    for i in range(n_walkers):
        p0[i,0] = np.random.uniform(0.95*peak, 1.05*peak)
        p0[i,1] = np.random.uniform(-rad_pix/2+rad_coord[0], rad_pix/2+rad_coord[0])
        p0[i,2] = np.random.uniform(-rad_pix/2+rad_coord[1], rad_pix/2+rad_coord[1])
    return p0

def c_p0(peak, rad_coord, rad_pix, rad_barea, total_flux, n_walkers):
    p0 = np.zeros((n_walkers, 4))
    source_area = rad_barea * total_flux / peak
    sigma = np.sqrt(source_area / (2*np.pi))
    vis_sigma = 1/(2*np.pi*sigma)
    for i in range(n_walkers):
        p0[i,0] = np.random.uniform(0.95*peak, 1.05*peak)
        p0[i,1] = np.random.uniform(-rad_pix/2+rad_coord[0], rad_pix/2+rad_coord[0])
        p0[i,2] = np.random.uniform(-rad_pix/2+rad_coord[1], rad_pix/2+rad_coord[1])
        p0[i,3] = np.random.uniform(0.95*vis_sigma, 1.05*vis_sigma)
    return p0

def g_p0(peak, rad_coord, rad_pix, rad_barea, total_flux, n_walkers):
    p0 = np.zeros((n_walkers, 6))
    source_area = rad_barea * total_flux / peak
    sigma = np.sqrt(source_area / (2*np.pi))
    vis_sigma = 1/(2*np.pi*sigma)
    for i in range(n_walkers):
        p0[i,0] = np.random.uniform(0.95*peak, 1.05*peak)
        p0[i,1] = np.random.uniform(-rad_pix/2+rad_coord[0], rad_pix/2+rad_coord[0])
        p0[i,2] = np.random.uniform(-rad_pix/2+rad_coord[1], rad_pix/2+rad_coord[1])
        p0[i,3] = np.random.uniform(0.95*vis_sigma, 1.05*vis_sigma)
        p0[i,4] = np.random.uniform(0, 1)
        p0[i,5] = np.random.uniform(-np.pi/2, np.pi/2)
    return p0

def d_p0(peak, rad_coord, rad_pix, rad_barea, total_flux, n_walkers):
    p0 = np.zeros((n_walkers, 4))
    source_area = rad_barea * total_flux / peak
    r = np.sqrt(source_area / (2*np.pi))
    vis_r = 1/(math.pi*r)
    for i in range(n_walkers):
        p0[i,0] = np.random.uniform(0.95*peak, 1.05*peak)
        p0[i,1] = np.random.uniform(-rad_pix/2+rad_coord[0], rad_pix/2+rad_coord[0])
        p0[i,2] = np.random.uniform(-rad_pix/2+rad_coord[1], rad_pix/2+rad_coord[1])
        p0[i,3] = np.random.uniform(0.95*vis_r, 1.05*vis_r)
    return p0

In [110]:
def p_prior(params):
    i0, l0, m0 = params
    if 0 < i0:
        return 0.0
    return -np.inf

def c_prior(params):
    s0, l0, m0, vis_sigma = params
    if 0 < s0 and 0 < vis_sigma:
        return 0.0
    return -np.inf

def g_prior(params):
    s0, l0, m0, vis_sigma, ratio, vis_theta = params
    if 0 < s0 and 0 < vis_sigma and 0 < ratio <= 1 and -np.pi/2 <= vis_theta <= np.pi/2:
        return 0.0
    return -np.inf

def d_prior(params):
    s0, l0, m0, vis_r = params
    if 0 < s0 and 0 < vis_r:
        return 0.0
    return -np.inf

In [111]:
def log_likelihood(model, re, im, u, v, w):
    return -0.5 * np.sum(w * ((re - model.real)**2 + (im - model.imag)**2))

In [112]:
P_PARAMS = ['i0', 'l0', 'm0']
C_PARAMS = ['i0', 'l0', 'm0', 'vis_sigma']
G_PARAMS = ['i0', 'l0', 'm0', 'vis_sigma', 'ratio', 'vis_theta']
D_PARAMS = ['i0', 'l0', 'm0', 'vis_r']

In [113]:
SOURCE_TYPES = {'p': [3, p_p0, p_prior, p_model, P_PARAMS], \
                'c': [4, c_p0, c_prior, c_model, C_PARAMS], \
                'g': [6, g_p0, g_prior, g_model, G_PARAMS], \
                'd': [4, d_p0, d_prior, d_model, D_PARAMS]}

In [114]:
def log_probability(params, sources, re, im, u, v, w):
    log_prior = 0.0
    model = 0.0
    i = 0
    for source in sources:
        n_params = SOURCE_TYPES[source][0]
        prior_func = SOURCE_TYPES[source][2]
        model_func = SOURCE_TYPES[source][3]
        source_params = params[i:i+n_params]
        lp = prior_func(source_params)
        if not np.isfinite(lp):
            return -np.inf
        log_prior += lp
        model += model_func(source_params, u, v)
        i += n_params
    log_likelihood_value = log_likelihood(model, re, im, u, v, w)
    return log_prior + log_likelihood_value

In [ ]:
def uv_fit(fits_file, sources, clean_output=True, corner_plot=True):
    # TODO: documentation
    '''
    Fit UV data from a FITS file with specified source types using MCMC.

    Parameters
    ----------
    fits_file (str): Path to the UV FITS file.
    sources (list): List of source types to fit. Each source type should be one of
                    'p' (point), 'c' (circular gaussian), 'g' (gaussian), 'd' (disk), or 'any' (try all and pick best fit).
    '''

    # Check input source types
    if len(sources) == 0:
        raise ValueError("No sources specified. Try specifying one or more sources of type \
                         'p' (point), 'c' (circular gaussian), 'g' (gaussian), 'd' (disk), or 'any' (try all and pick best fit).")
    for source in sources:
        if source not in SOURCE_TYPES and source != 'any':
            raise ValueError(f"Source type '{source}' is not recognized. Source type must be one of the following: \
                            'p' (point), 'c' (circular gaussian), 'g' (gaussian), 'd' (disk), or 'any' (try all and pick best fit).")

    # Extract data from fits file
    file = fits.open(fits_file)
    cdelt1 = file[0].header['CDELT1']
    cunit1 = file[0].header['CUNIT1']
    naxis1 = file[0].header['NAXIS1']
    data = file[1].data

    summ = summary(fits_file, plot=False)
    bmaj = file[0].header['BMAJ'] # cunit1
    bmin = file[0].header['BMIN'] # cunit1
    rad_bmaj = Angle(bmaj, cunit1).to(units.radian).value
    rad_bmin = Angle(bmin, cunit1).to(units.radian).value
    rad_barea = np.pi * rad_bmaj * rad_bmin / (4 * np.log(2))
    rad_pix = float(Angle(cdelt1, cunit1).to(units.radian).value)
    int_peaks = summ['int_peak_val']
    int_coords = summ['int_peak_coord']
    ext_peaks = summ['ext_peak_val']
    ext_coords = summ['ext_peak_coord']

    int_info = list(zip(int_peaks, int_coords))
    if type(ext_peaks) is list:
        ext_info = list(zip(ext_peaks, ext_coords))
    else:
        ext_info = []
    all_peaks = int_info + ext_info # list of tuples (peak_value, (l_coord, m_coord))
    all_peaks.sort(reverse=True) # sort by peak value

    vis = np.array(data)
    freq_bin, u, v, re, im, w = [], [], [], [], [], []
    for row in vis:
        freq_bin_data, u_data, v_data, re_data, im_data, w_data = row
        freq_bin.append(int(freq_bin_data))
        u.append(int(u_data))
        v.append(int(v_data))
        re.append(float(re_data/w_data))
        im.append(float(im_data/w_data))
        w.append(float(w_data))

    # Adding in conjugate half of data
    freq_bin *= 2
    neg_u = [-1 * val for val in u]
    u += neg_u
    neg_v = [-1 * val for val in v]
    v += neg_v
    re *= 2
    neg_im = [-1 * val for val in im]
    im += neg_im
    w *= 2

    freq_bin = np.array(freq_bin)
    u = np.array(u)
    v = np.array(v)
    re = np.array(re)
    im = np.array(im)
    w = np.array(w)

    file.close() # good practice

    total_flux = np.max((re**2 + im**2)**(1/2))

    # All possible permutations
    n_sources = len(sources)
    sample_space = list(SOURCE_TYPES.keys()) * n_sources
    all_permutations = list(itertools.permutations(sample_space, n_sources))

    for i in range(n_sources):
        if sources[i] != 'any':
            all_permutations = [p for p in all_permutations if p[i] == sources[i]] # remove unwanted permutations

    all_results = []
    for permutation in all_permutations:
        # Calculate n_dim and n_walkers
        n_dim = 0
        for i in range(n_sources):
            source = permutation[i]
            n_dim += SOURCE_TYPES[source][0]
        n_walkers = 2 * n_dim

        # Initial guesses
        n_params = 0
        for i in range(n_sources):
            source = permutation[i]
            peak = all_peaks[i][0]
            coord0 = all_peaks[i][1]
            rad_coord = (float(Angle(coord0[0], units.arcsec).to(units.radian).value), float(Angle(coord0[1], units.arcsec).to(units.radian).value))
            if i == 0:
                p0 = SOURCE_TYPES[source][1](peak, rad_coord, rad_pix, rad_barea, total_flux, n_walkers)
            else:
                p0 = np.append(p0, SOURCE_TYPES[source][1](peak, rad_coord, rad_pix, rad_barea, total_flux, n_walkers), axis=0)
            n_params += SOURCE_TYPES[source][0]

        # Set up and run MCMC
        n_steps = 100
        sampler = emcee.EnsembleSampler(n_walkers, n_dim, log_probability, args=(permutation, re, im, u, v, w))
        try:
            state = sampler.run_mcmc(p0, n_steps)
        except emcee.autocorr.AutocorrError:
            pass
        tau = sampler.get_autocorr_time(quiet=True)
        int_tau = math.ceil(np.nanmax(tau)/1)
        steps_to_50_tau = int_tau * 50 - n_steps
        sampler.run_mcmc(state, steps_to_50_tau)
        chain = sampler.get_chain(discard = int_tau * 10, flat=True)

        # Find parameter estimates and uncertainties and calculate chi2
        result = {}
        model = 0.0
        for i in range(n_sources):
            source = permutation[i]
            n_params = SOURCE_TYPES[source][0]
            source_chain = chain[:, i*n_params:(i+1)*n_params]
            source_result = {'type': source}
            temp_medians = [] # to store medians for chi2 calculation
            for j in range(n_params):
                samples = source_chain[:, j]
                samples_med = np.median(samples)
                samples_sd = np.nanstd(samples)
                param_name = SOURCE_TYPES[source][4][j]
                source_result[param_name] = (float(samples_med), float(samples_sd))
                temp_medians.append(samples_med)
            model += SOURCE_TYPES[source][3](temp_medians, u, v)
            result[f'source_{i+1}'] = source_result
        chi2 = float(np.sum(w * ((re - model.real)**2 + (im - model.imag)**2)))

        all_results.append({'permutation': permutation, 'n_params': n_params, 'result': result, 'chi2': chi2})

    # Bayesian Information Criterion
    n = len(re)
    for permutation_info in all_results:
        k = permutation_info['n_params']
        chi2 = permutation_info['chi2']
        bic = k * np.log(n) + chi2
        permutation_info['BIC'] = float(bic)
    all_results.sort(key=lambda x: x['BIC']) # lowest to highest BIC

    if clean_output:
        for permutation_info in all_results:
            result = permutation_info['result']
            for i in range(n_sources):
                source_key = f'source_{i+1}'
                source_result = result[source_key]
                source_type = source_result['type']
                source_params = SOURCE_TYPES[source_type][4]

                # TODO: CLEAN OUTPUT
                for param in source_params:
                    if param == 'i0':
                        if source_type != 'p': # convert flux to peak

    return all_results

    # TODO: TEST
    # TODO: CORNER PLOTS
    # TODO: CASE: NUMBER OF SOURCES > NUMBER OF PEAKS (AS DETECTED BY summary())?
    # TODO: USE BETTER INITIAL GUESSES TO RE-TRY FITTING IF NEEDED?
    # TODO: CASE: GAUSSIAN MODEL TO FIT CIRCULAR GAUSSIAN: ISSUES WITH THETA AND RATIO?

In [116]:
uv_fit('../data/uv_test/3c84.uvfits', sources=['any'])

The chain is shorter than 50 times the integrated autocorrelation time for 3 parameter(s). Use this estimate with caution and run a longer chain!
N/50 = 2;
tau: [13.51431961  8.79357951 12.87136488]
The chain is shorter than 50 times the integrated autocorrelation time for 4 parameter(s). Use this estimate with caution and run a longer chain!
N/50 = 2;
tau: [14.5733453  12.59022055 12.34808799 14.31374999]
The chain is shorter than 50 times the integrated autocorrelation time for 6 parameter(s). Use this estimate with caution and run a longer chain!
N/50 = 2;
tau: [13.86046118 12.8727076  12.76045781 13.53490448 10.52166037  7.2143768 ]
The chain is shorter than 50 times the integrated autocorrelation time for 4 parameter(s). Use this estimate with caution and run a longer chain!
N/50 = 2;
tau: [10.34251283  6.90590919  8.9091232  13.80267211]


['i0', 'l0', 'm0']
['i0', 'l0', 'm0', 'vis_r']
['i0', 'l0', 'm0', 'vis_sigma']
['i0', 'l0', 'm0', 'vis_sigma', 'ratio', 'vis_theta']


[{'permutation': ('p',),
  'n_params': 3,
  'result': {'source_1': {'type': 'p',
    'i0': (2.424722876600254, 0.03350951698264572),
    'l0': (4.1623022175146444e-08, 5.505834051882226e-08),
    'm0': (-6.078765741828306e-08, 8.10855025323127e-08)}},
  'chi2': 7471.800932431279,
  'BIC': 7493.433387791695},
 {'permutation': ('d',),
  'n_params': 4,
  'result': {'source_1': {'type': 'd',
    'i0': (2.7945091365738977, 0.1467638878038092),
    'l0': (4.643260067643463e-08, 3.554921782762968e-08),
    'm0': (-5.7544241597692265e-08, 3.467432062935819e-08),
    'vis_r': (71659.59471808543, 14677.592293205587)}},
  'chi2': 7507.535969165142,
  'BIC': 7536.379242979031},
 {'permutation': ('c',),
  'n_params': 4,
  'result': {'source_1': {'type': 'c',
    'i0': (3.527559449492605, 0.10368412405777096),
    'l0': (3.349048289513766e-07, 3.557317778564687e-07),
    'm0': (5.236774458128237e-07, 1.2350691219654091e-06),
    'vis_sigma': (33187.83017395819, 1591.1867423879771)}},
  'chi2': 21679